# Phase 2 – L1-Based Feature Selection

This notebook implements **Phase 2** of the CS 6735 project.  
L1 feature selection (`SelectFromModel` + `LinearSVC(penalty='l1')`) is applied **inside each fold** to prevent data leakage.

Outputs:
- `l1_results.json` – per-fold accuracy, F1, selected features
- `Results.xlsx` – Sheet **'After FS-DR'** filled in

## 1. Imports

In [1]:
import csv
import json
from pathlib import Path

import numpy as np
from openpyxl import load_workbook

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.tree import DecisionTreeClassifier

## 2. Project Paths & Constants

In [2]:
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data' / 'processed' / 'Datasets').exists():
            return candidate
    raise FileNotFoundError('Could not find project root containing data/processed/Datasets')

PROJECT_ROOT   = find_project_root(Path.cwd())
DATASETS_ROOT  = PROJECT_ROOT / 'data' / 'processed' / 'Datasets'
TARGET_COLUMN  = 'Label'
RANDOM_STATE   = 42

print('PROJECT_ROOT :', PROJECT_ROOT)
print('DATASETS_ROOT:', DATASETS_ROOT)

PROJECT_ROOT : /Users/sultanarazia/Desktop/UNB/Machine Learning/Project/L1-feature-selection
DATASETS_ROOT: /Users/sultanarazia/Desktop/UNB/Machine Learning/Project/L1-feature-selection/data/processed/Datasets


## 3. Classifiers & Outer CV  
*(Same 5 classifiers and 10-fold outer CV as Phase 1)*

In [3]:
classifiers = {
    'SVM':          SVC(random_state=RANDOM_STATE),
    'kNN':          KNeighborsClassifier(),
    'DecisionTree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(random_state=RANDOM_STATE),
    'MLP':          MLPClassifier(random_state=RANDOM_STATE, max_iter=1000),
}

outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=5,  shuffle=True, random_state=RANDOM_STATE)

# L1 C values to search over (inner loop)
L1_C_GRID = [0.001, 0.01, 0.1, 1.0]

print('Classifiers:', list(classifiers.keys()))
print('Outer CV   :', outer_cv)
print('Inner CV   :', inner_cv)
print('L1 C grid  :', L1_C_GRID)

Classifiers: ['SVM', 'kNN', 'DecisionTree', 'RandomForest', 'MLP']
Outer CV   : StratifiedKFold(n_splits=10, random_state=42, shuffle=True)
Inner CV   : StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
L1 C grid  : [0.001, 0.01, 0.1, 1.0]


## 4. Data Loader

In [4]:
def load_train_dataset(dataset_id, datasets_root=DATASETS_ROOT, target_column=TARGET_COLUMN):
    train_csv = datasets_root / str(dataset_id) / 'train.csv'
    if not train_csv.exists():
        raise FileNotFoundError(f'Missing file: {train_csv}')

    with train_csv.open(newline='') as f:
        reader = csv.DictReader(f)
        if reader.fieldnames is None:
            raise ValueError(f'No header found in {train_csv}')
        if target_column not in reader.fieldnames:
            raise ValueError(f"Target column '{target_column}' not found in {train_csv}")

        feature_names = [c for c in reader.fieldnames if c != target_column]
        rows = list(reader)

    def to_float_or_nan(value):
        if value is None:
            return np.nan
        value = value.strip()
        return np.nan if value == '' else float(value)

    X = np.array([[to_float_or_nan(r[c]) for c in feature_names] for r in rows], dtype=float)
    y = np.array([r[target_column] for r in rows])
    return X, y, feature_names

## 5. L1 Feature Selection Helper

**Design rules (no data leakage):**
- `SimpleImputer` → `StandardScaler` → `SelectFromModel(LinearSVC, penalty='l1')` → classifier
- All steps fit **only on the training fold**, then applied to the validation fold.
- The best L1 `C` is chosen via an inner 5-fold CV on the training fold (nested CV).

In [5]:
def select_best_l1_C(X_train, y_train, c_grid=L1_C_GRID):
    """
    Choose the best L1 C via inner cross-validation on the training fold.
    Uses a simple LinearSVC pipeline for selection scoring.
    Returns the C value that yields the best mean accuracy in the inner loop.
    """
    best_C, best_score = c_grid[0], -1

    for C in c_grid:
        inner_pipe = Pipeline([
            ('imputer',  SimpleImputer(strategy='median')),
            ('scaler',   StandardScaler()),
            ('selector', SelectFromModel(
                LinearSVC(C=C, penalty='l1', dual=False,
                          max_iter=10000, random_state=RANDOM_STATE),
                prefit=False
            )),
            ('clf', LinearSVC(C=1.0, max_iter=10000, random_state=RANDOM_STATE)),
        ])

        scores = []
        for tr_idx, val_idx in inner_cv.split(X_train, y_train):
            try:
                inner_pipe.fit(X_train[tr_idx], y_train[tr_idx])
                scores.append(accuracy_score(y_train[val_idx],
                                             inner_pipe.predict(X_train[val_idx])))
            except Exception:
                scores.append(0.0)   # selector might zero-out all features

        mean_score = np.mean(scores)
        if mean_score > best_score:
            best_score, best_C = mean_score, C

    return best_C


def build_l1_pipeline(clf_name, clf_model, l1_C):
    """
    Build the full pipeline with L1 selection baked in.
    Distance-sensitive classifiers (SVM, kNN, MLP) get a second scaler
    after selection so their inputs remain standardised.
    """
    steps = [
        ('imputer',  SimpleImputer(strategy='median')),
        ('scaler',   StandardScaler()),
        ('selector', SelectFromModel(
            LinearSVC(C=l1_C, penalty='l1', dual=False,
                      max_iter=10000, random_state=RANDOM_STATE),
            prefit=False
        )),
    ]

    if clf_name in ['SVM', 'kNN', 'MLP']:
        steps.append(('scaler2', StandardScaler()))

    steps.append(('clf', clone(clf_model)))
    return Pipeline(steps)


print('Helper functions defined.')

Helper functions defined.


## 6. Phase 2 Experiment Loop

For every dataset × classifier × fold:
1. Split into train / validation.
2. Find best L1 `C` via inner CV on training fold.
3. Build & fit the L1 pipeline on the training fold.
4. Predict on the validation fold.
5. Record accuracy, macro-F1, number of selected features, feature names, and best C.

In [ ]:
l1_results = {}

for dataset_id in range(1, 17):
    X, y, feature_names = load_train_dataset(dataset_id)
    feature_names_arr = np.array(feature_names)

    l1_results[dataset_id] = {}
    print(f'\n=== Dataset {dataset_id} | shape={X.shape} ===')

    for clf_name, clf_model in classifiers.items():
        fold_rows = []

        for fold_idx, (train_idx, valid_idx) in enumerate(
                outer_cv.split(X, y), start=1):

            X_train, X_valid = X[train_idx], X[valid_idx]
            y_train, y_valid = y[train_idx], y[valid_idx]

            # --- Step 1: find best L1 C via inner CV (training fold only) ---
            best_C = select_best_l1_C(X_train, y_train)

            # --- Step 2: build & fit pipeline ---
            pipeline = build_l1_pipeline(clf_name, clf_model, best_C)
            pipeline.fit(X_train, y_train)

            # --- Step 3: evaluate ---
            y_pred = pipeline.predict(X_valid)
            acc    = accuracy_score(y_valid, y_pred)
            mf1    = f1_score(y_valid, y_pred, average='macro', zero_division=0)

            # --- Step 4: record selected features ---
            selector         = pipeline.named_steps['selector']
            selected_mask    = selector.get_support()
            selected_features = feature_names_arr[selected_mask].tolist()
            n_selected       = len(selected_features)

            fold_rows.append({
                'fold':             fold_idx,
                'accuracy':         round(acc, 6),
                'macro_f1':         round(mf1, 6),
                'n_selected':       n_selected,
                'selected_features': selected_features,
                'best_l1_C':        best_C,
            })

        l1_results[dataset_id][clf_name] = fold_rows

        # Per-classifier summary
        acc_mean = np.mean([r['accuracy'] for r in fold_rows])
        acc_std  = np.std( [r['accuracy'] for r in fold_rows])
        f1_mean  = np.mean([r['macro_f1'] for r in fold_rows])
        f1_std   = np.std( [r['macro_f1'] for r in fold_rows])
        n_mean   = np.mean([r['n_selected'] for r in fold_rows])
        print(f'  {clf_name:12s} | '
              f'acc={acc_mean:.4f}±{acc_std:.4f} | '
              f'macro_f1={f1_mean:.4f}±{f1_std:.4f} | '
              f'avg_features={n_mean:.1f}')

print('\nPhase 2 experiment complete.')
output_json = PROJECT_ROOT / 'data' / 'processed' / 'l1_results.json'

with output_json.open('w') as f:
    json.dump(l1_results, f, indent=2)

print(f'Saved l1_results to: {output_json}')


=== Dataset 1 | shape=(9583, 19) ===


/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 6 members, which is less than n_splits=10.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of i

  SVM          | acc=0.9518±0.0072 | macro_f1=0.7803±0.1156 | avg_features=12.8


/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 6 members, which is less than n_splits=10.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of i

  kNN          | acc=0.9784±0.0049 | macro_f1=0.8128±0.1188 | avg_features=12.8


...

/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 8 members, which is less than n_splits=10.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of i

  RandomForest | acc=0.9995±0.0010 | macro_f1=0.9995±0.0009 | avg_features=12.8


/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 8 members, which is less than n_splits=10.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of i

  MLP          | acc=0.9874±0.0057 | macro_f1=0.9599±0.0762 | avg_features=12.8

=== Dataset 9 | shape=(8330, 265) ===


/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


## 7. Save Results to JSON

In [ ]:
output_json = PROJECT_ROOT / 'data' / 'processed' / 'l1_results.json'

with output_json.open('w') as f:
    json.dump(l1_results, f, indent=2)

print(f'Saved l1_results to: {output_json}')

## 8. Export to Excel – Sheet "After FS-DR"

Fills in the same column layout as Sheet 1 (Before FS-DR) but also writes:
- number of selected features
- selected feature names
- best L1 C parameter

In [ ]:
results_xlsx_path = DATASETS_ROOT / 'Results.xlsx'
wb = load_workbook(results_xlsx_path)
ws = wb['After FS-DR']

# Columns: acc | f1 | params  (same layout as Sheet 1)
column_map = {
    'SVM':          ('B', 'C', 'L'),
    'kNN':          ('D', 'E', 'M'),
    'DecisionTree': ('F', 'G', 'N'),
    'RandomForest': ('H', 'I', 'O'),
    'MLP':          ('J', 'K', 'P'),
}

for dataset_id in range(1, 17):
    start_row = 3 + (dataset_id - 1) * 12   # same row formula as Sheet 1

    for clf_name, rows in l1_results[dataset_id].items():
        acc_col, f1_col, param_col = column_map[clf_name]

        for row in rows:
            excel_row = start_row + row['fold'] - 1
            ws[f'{acc_col}{excel_row}'] = row['accuracy']
            ws[f'{f1_col}{excel_row}']  = row['macro_f1']
            ws[f'{param_col}{excel_row}'] = str({
                'l1_C':               row['best_l1_C'],
                'n_features_selected': row['n_selected'],
                'selected_features':  row['selected_features'],
            })

wb.save(results_xlsx_path)
print(f'Exported Phase 2 results to: {results_xlsx_path}')

## 9. Summary Table (mean ± std for report)

In [ ]:
print(f"{'Dataset':<10} {'Classifier':<14} {'Accuracy':>18} {'Macro-F1':>18} {'Avg Features':>14}")
print('-' * 76)

for dataset_id in range(1, 17):
    for clf_name in classifiers.keys():
        rows     = l1_results[dataset_id][clf_name]
        acc_vals = [r['accuracy'] for r in rows]
        f1_vals  = [r['macro_f1'] for r in rows]
        n_vals   = [r['n_selected'] for r in rows]

        acc_str = f'{np.mean(acc_vals):.4f} ± {np.std(acc_vals):.4f}'
        f1_str  = f'{np.mean(f1_vals):.4f} ± {np.std(f1_vals):.4f}'
        n_str   = f'{np.mean(n_vals):.1f}'

        print(f'{dataset_id:<10} {clf_name:<14} {acc_str:>18} {f1_str:>18} {n_str:>14}')